# 02 · Modelo base v2 DERM12345 — sin atajo de recorte y umbral por validación cruzada

**Fase A del plan, segunda iteración.** Corrige los dos problemas de la v1
(`01_baseline_derm12345.ipynb`):

| | v1 · resultado en test | Problema |
|---|---|---|
| AUC | 0,976 (IC95 0,95–1,00) | — |
| Sensibilidad | **0,75** (21/28) | Por debajo del objetivo (≥ 0,90) |
| Especificidad | 0,993 (1 falsa alarma de 153) | Umbral demasiado alto (0,793) |

1. **Atajo de recorte.** En train, el 69 % de los melanomas son imágenes recortadas a mano
   (tamaño irregular) frente al 7 % de los nevus; en test, solo el 11 % de los melanomas.
   El modelo podía aprender "recortada = melanoma". → Ahora **todas** las imágenes pasan por
   el mismo recorte cuadrado centrado en la lesión (`ms2-procesamiento/app/preprocessing.py`,
   el mismo código que usará `/analyze`).
2. **Umbral inestable.** Se elegía con ~18 melanomas de validación. → Ahora con
   **validación cruzada de 5 particiones por paciente** sobre todo el train: el umbral sale de
   las predicciones *out-of-fold* de los 108 melanomas. Objetivo subido a **0,95** en
   validación para dejar margen y acercarse a ≥ 0,90 en test.
3. **Métricas por subgrupo** (imagen estándar / recortada) para comprobar que el atajo ya no pesa.

El test sigue siendo el del paper y **solo se mira al final**, una vez.

### Cómo ejecutarlo en Colab
1. Abre <https://colab.research.google.com/github/Mangelits/AcralisisPodoScan/blob/main/notebooks/02_baseline_v2_derm12345.ipynb>
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
3. *Entorno de ejecución → Ejecutar todas*. Entrena 6 modelos (5 particiones + final):
   cuenta con ~20–30 min en una T4.

In [ ]:
# ---- Configuración -------------------------------------------------------
REPO_URL = "https://github.com/Mangelits/AcralisisPodoScan.git"
BRANCH = "main"
USE_DRIVE = True          # imágenes y modelos en Google Drive

SEED = 42
IMG_SIZE = 224            # resolución nativa de EfficientNet-B0
CACHE_SIZE = 384          # lado de los recortes cuadrados en caché
CROP_MARGIN = 1.5         # lado del recorte = 1,5 × lado mayor de la lesión
EPOCHS = 20               # fijo: sin early stopping, se usa siempre la última época
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
N_FOLDS = 5
TARGET_SENSITIVITY = 0.95 # en validación cruzada; el objetivo en test es ≥ 0,90

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/AcralisisPodoScan").exists():
        !git clone -q -b {BRANCH} {REPO_URL} /content/AcralisisPodoScan
    %cd /content/AcralisisPodoScan
    !git pull -q
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
if IN_COLAB and USE_DRIVE:
    STORE = Path("/content/drive/MyDrive/acralis")
    RAW_DIR, MODELS_DIR = STORE / "data" / "raw" / "derm12345", STORE / "models"
else:  # en local, todo dentro de data/ (ignorado por git)
    RAW_DIR, MODELS_DIR = ROOT / "data" / "raw" / "derm12345", ROOT / "data" / "processed" / "models"
CACHE_DIR = ROOT / "data" / "processed" / f"derm12345_lesion_{CACHE_SIZE}_m{CROP_MARGIN}"
for d in (RAW_DIR, CACHE_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Preprocesamiento compartido con MS2 (/analyze usará exactamente el mismo).
sys.path.insert(0, str(ROOT / "ms2-procesamiento"))
from app.preprocessing import lesion_crop
print("ROOT:", ROOT, "\nRAW_DIR:", RAW_DIR, "\nMODELS_DIR:", MODELS_DIR)

## 1 · Datos

In [ ]:
!python scripts/datasets/download_derm12345.py --out "{RAW_DIR}"

# Si la descarga falla, parar aquí (si no, todo lo siguiente sale vacío).
acral_csv = RAW_DIR / "acral.csv"
n_rows = sum(1 for _ in open(acral_csv)) - 1 if acral_csv.exists() else 0
assert n_rows == 896, (
    f"acral.csv tiene {n_rows} de 896 imágenes. Revisa los errores (✗) de la salida "
    "de arriba y vuelve a ejecutar esta celda: la descarga es reanudable."
)
print("✓ 896 imágenes acrales listas")

### Recorte centrado en la lesión

Cada imagen original se recorta a un cuadrado centrado en la lesión y se guarda a
384×384. Se anota el tamaño original (para los subgrupos) y si se detectó la lesión
o se usó el cuadrado central como respaldo.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

def cache_crop(src: Path) -> dict:
    dst = CACHE_DIR / src.name
    meta_path = dst.with_suffix(".json")
    if dst.exists() and meta_path.exists():
        return json.loads(meta_path.read_text())
    img = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
    crop, info = lesion_crop(np.asarray(img), margin=CROP_MARGIN)
    Image.fromarray(crop).resize((CACHE_SIZE, CACHE_SIZE), Image.LANCZOS).save(dst, quality=95)
    meta = {"isic_id": src.stem, "orig_w": img.width, "orig_h": img.height,
            "lesion_found": info.found, "crop_side": info.box[2]}
    meta_path.write_text(json.dumps(meta))
    return meta

sources = sorted((RAW_DIR / "images").glob("*.jpg"))
with ThreadPoolExecutor(8) as pool:
    crop_meta = pd.DataFrame(list(pool.map(cache_crop, sources)))
print(f"{len(crop_meta)} recortes en {CACHE_DIR}")
print(f"Lesión detectada en {crop_meta.lesion_found.mean():.0%}; el resto usa el cuadrado central.")

In [ ]:
df = pd.read_csv(RAW_DIR / "acral.csv").merge(crop_meta, on="isic_id")
df["target"] = (df["malignancy"] == "malignant").astype(int)   # 1 = sospechoso de melanoma
df["path"] = [str(CACHE_DIR / Path(f).name) for f in df["file"]]

# Subgrupo: tamaños de cámara habituales (≥10 imágenes) = "estándar"; el resto, recortes a mano.
size = df.orig_w.astype(str) + "x" + df.orig_h.astype(str)
common = size.value_counts()
df["subgroup"] = np.where(size.map(common) >= 10, "estándar", "recortada")

train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
assert not set(train_df.patient_id) & set(test_df.patient_id)

pd.crosstab([df.split, df.subgroup], df.target.map({0: "benigno", 1: "melanoma"}))

**Inspección visual.** Comprueba que los recortes están centrados en la lesión en los
cuatro grupos. Si en muchos se pierde la lesión, ajusta `CROP_MARGIN` o avisa: la
heurística (Otsu) se puede afinar.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
groups = [(s, t) for s in ("estándar", "recortada") for t in (0, 1)]
fig, axes = plt.subplots(len(groups), 8, figsize=(16, 8.5))
for row, (s, t) in zip(axes, groups):
    sub = df[(df.subgroup == s) & (df.target == t)]
    picks = sub.sample(min(8, len(sub)), random_state=SEED)
    for ax in row:
        ax.axis("off")
    for ax, (_, r) in zip(row, picks.iterrows()):
        ax.imshow(Image.open(r.path))
        ax.set_title(("" if r.lesion_found else "⚠ central · ") + r.label, fontsize=8)
    row[0].text(-0.1, 0.5, f"{s}\n{'melanoma' if t else 'benigno'}", transform=row[0].transAxes,
                ha="right", va="center", fontsize=10)
plt.tight_layout(); plt.show()

## 2 · Dataset, aumento de datos y entrenamiento

Las imágenes ya son cuadradas y centradas en la lesión, así que el aumento usa recortes
aleatorios suaves (70–100 % del área) en vez de los agresivos de la v1.

In [ ]:
import copy
import random
import time

import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

def seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if AMP else "")
if not AMP:
    print("⚠️ Sin GPU: activa una en Entorno de ejecución → Cambiar tipo de entorno.")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(180),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LesionDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tf):
        self.paths, self.targets, self.tf = frame["path"].tolist(), frame["target"].tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), torch.tensor(self.targets[i], dtype=torch.float32)

def make_loader(frame: pd.DataFrame, train: bool) -> DataLoader:
    return DataLoader(LesionDataset(frame, train_tf if train else eval_tf), batch_size=BATCH_SIZE,
                      shuffle=train, drop_last=train, num_workers=2, pin_memory=AMP)

@torch.no_grad()
def predict(model, frame: pd.DataFrame) -> np.ndarray:
    model.eval()
    probs = []
    for x, _ in make_loader(frame, train=False):
        with torch.autocast(DEVICE.type, enabled=AMP):
            logits = model(x.to(DEVICE, non_blocking=True)).squeeze(1)
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(probs)

def train_model(train_frame: pd.DataFrame, monitor_frame: pd.DataFrame | None = None, tag: str = ""):
    """Entrena EPOCHS épocas y devuelve el modelo de la última época.

    monitor_frame solo se usa para mostrar la AUC por época, nunca para elegir época.
    """
    seed_everything(SEED)
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 1)
    model = model.to(DEVICE)
    dl = make_loader(train_frame, train=True)
    pos_weight = torch.tensor((train_frame.target == 0).sum() / (train_frame.target == 1).sum(), device=DEVICE)
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS,
                                                    steps_per_epoch=len(dl), pct_start=0.15)
    scaler = torch.amp.GradScaler(enabled=AMP)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0, running = time.time(), 0.0
        for x, y in dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=AMP):
                loss = criterion(model(x).squeeze(1), y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item() * len(y)
        row = {"epoch": epoch, "loss": running / (len(dl) * BATCH_SIZE)}
        if monitor_frame is not None:
            row["val_auc"] = roc_auc_score(monitor_frame.target, predict(model, monitor_frame))
        history.append(row)
        auc_txt = f" · val AUC {row['val_auc']:.3f}" if "val_auc" in row else ""
        print(f"{tag} época {epoch:2d} · loss {row['loss']:.4f}{auc_txt} · {time.time() - t0:.0f}s")
    return model, history

## 3 · Validación cruzada por paciente → predicciones *out-of-fold*

Cada imagen de train recibe una predicción de un modelo que **no la vio** al entrenar.
Con esas 715 predicciones (108 melanomas) se elige el umbral.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for k, (_, va_idx) in enumerate(sgkf.split(train_df, train_df.target, groups=train_df.patient_id)):
    train_df.loc[va_idx, "fold"] = k
assert (train_df.groupby("patient_id").fold.nunique() == 1).all()   # cada paciente en un solo fold
print(pd.crosstab(train_df.fold, train_df.target.map({0: "benigno", 1: "melanoma"})))

train_df["oof"] = np.nan
fold_histories = []
for k in range(N_FOLDS):
    tr, va = train_df[train_df.fold != k], train_df[train_df.fold == k]
    model, hist = train_model(tr, monitor_frame=va, tag=f"[fold {k}]")
    train_df.loc[va.index, "oof"] = predict(model, va)
    fold_histories.append(pd.DataFrame(hist).assign(fold=k))
    print(f"[fold {k}] AUC final {roc_auc_score(va.target, train_df.loc[va.index, 'oof']):.3f}\n")
    del model; torch.cuda.empty_cache()

In [ ]:
fh = pd.concat(fold_histories)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for k, g in fh.groupby("fold"):
    ax[0].plot(g.epoch, g.loss, label=f"fold {k}"); ax[1].plot(g.epoch, g.val_auc)
ax[0].set(title="Pérdida de entrenamiento", xlabel="época"); ax[0].legend(fontsize=8)
ax[1].set(title="AUC en la partición de validación", xlabel="época", ylim=(0.5, 1))
plt.tight_layout(); plt.show()

## 4 · Umbral a partir de las predicciones *out-of-fold*

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve

def threshold_for_sensitivity(y, p, target):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr >= target)])

def summarize(y, p, thr) -> dict:
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"n": len(y), "melanomas": int(y.sum()),
            "auc": roc_auc_score(y, p) if 0 < y.sum() < len(y) else float("nan"),
            "sensibilidad": tp / (tp + fn) if tp + fn else float("nan"),
            "especificidad": tn / (tn + fp) if tn + fp else float("nan"),
            "tp": int(tp), "fn": int(fn), "fp": int(fp), "tn": int(tn)}

def by_subgroup(frame: pd.DataFrame, pcol: str, thr: float) -> pd.DataFrame:
    rows = {"todas": summarize(frame.target.values, frame[pcol].values, thr)}
    for s, g in frame.groupby("subgroup"):
        rows[s] = summarize(g.target.values, g[pcol].values, thr)
    return pd.DataFrame(rows).T

threshold = threshold_for_sensitivity(train_df.target, train_df.oof, TARGET_SENSITIVITY)
print(f"Umbral elegido (sensibilidad ≥ {TARGET_SENSITIVITY:.0%} out-of-fold): {threshold:.3f}")

# ¿Cuánto predice el atajo por sí solo? (AUC de "es recortada" como único predictor)
shortcut = (train_df.subgroup == "recortada").astype(int)
print(f"AUC del atajo 'recortada' por sí solo (train): {roc_auc_score(train_df.target, shortcut):.3f}")
by_subgroup(train_df, "oof", threshold).round(3)

## 5 · Modelo final y evaluación en test (una sola vez)

El modelo final se entrena con **todo** el train y las mismas épocas; se le aplica el
umbral de la validación cruzada sin tocarlo.

In [ ]:
final_model, final_history = train_model(train_df, tag="[final]")
test_df["p"] = predict(final_model, test_df)

y_test, p_test = test_df.target.values, test_df.p.values
pred = (p_test >= threshold).astype(int)

def bootstrap_ci(fn_metric, n=2000):
    rng, vals = np.random.default_rng(SEED), []
    for _ in range(n):
        i = rng.integers(0, len(y_test), len(y_test))
        if y_test[i].min() != y_test[i].max():
            vals.append(fn_metric(y_test[i], p_test[i], pred[i]))
    return np.percentile(vals, [2.5, 97.5])

metrics = summarize(y_test, p_test, threshold)
tp, fn, fp, tn = metrics["tp"], metrics["fn"], metrics["fp"], metrics["tn"]
metrics["vpp"] = tp / (tp + fp) if tp + fp else float("nan")
metrics["vpn"] = tn / (tn + fn) if tn + fn else float("nan")
ci = {
    "auc": bootstrap_ci(lambda y, p, _: roc_auc_score(y, p)),
    "sensibilidad": bootstrap_ci(lambda y, _, q: q[y == 1].mean()),
    "especificidad": bootstrap_ci(lambda y, _, q: 1 - q[y == 0].mean()),
}
V1 = {"auc": 0.976, "sensibilidad": 0.750, "especificidad": 0.993, "vpp": 0.955, "vpn": 0.956}

print(f"Umbral (de la validación cruzada): {threshold:.3f}")
print(f"\nTEST · {len(y_test)} imágenes ({int(y_test.sum())} melanomas)        v1")
for k in ("auc", "sensibilidad", "especificidad", "vpp", "vpn"):
    extra = f"  (IC95 {ci[k][0]:.2f}–{ci[k][1]:.2f})" if k in ci else " " * 18
    print(f"  {k:<14}{metrics[k]:.3f}{extra}   {V1[k]:.3f}")
print(f"\nMelanomas detectados {tp}/{tp + fn}  ·  falsas alarmas {fp}/{fp + tn}   (v1: 21/28 · 1/153)")

In [ ]:
print("Test por subgrupo:")
display(by_subgroup(test_df, "p", threshold).round(3))

print("\nMelanomas de test, de menor a mayor probabilidad (los primeros son los que se escapan):")
mel = test_df[test_df.target == 1].sort_values("p")
mel = mel.assign(detectado=np.where(mel.p >= threshold, "sí", "NO"),
                 tamaño=mel.orig_w.astype(str) + "×" + mel.orig_h.astype(str))
display(mel[["isic_id", "label", "subgroup", "tamaño", "lesion_found", "p", "detectado"]].round(3))

In [ ]:
fpr_t, tpr_t, _ = roc_curve(y_test, p_test)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(fpr_t, tpr_t, label=f"AUC {metrics['auc']:.3f}")
ax[0].plot([0, 1], [0, 1], ls="--", c="grey")
ax[0].scatter(1 - metrics["especificidad"], metrics["sensibilidad"], c="red", zorder=3, label="umbral")
ax[0].set(title="Curva ROC (test)", xlabel="1 − especificidad", ylabel="sensibilidad"); ax[0].legend()
cm = np.array([[tn, fp], [fn, tp]])
ax[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[1].text(j, i, v, ha="center", va="center", fontsize=14)
ax[1].set(title="Matriz de confusión (test)", xticks=[0, 1], yticks=[0, 1],
          xticklabels=["pred. benigno", "pred. sospechoso"], yticklabels=["benigno", "melanoma"])
for t, name in ((0, "benigno"), (1, "melanoma")):
    ax[2].hist(p_test[y_test == t], bins=20, range=(0, 1), alpha=0.6, label=name)
ax[2].axvline(threshold, c="red", ls="--", label="umbral")
ax[2].set(title="Probabilidades en test", xlabel="p(melanoma)"); ax[2].legend()
plt.tight_layout(); plt.show()

## 6 · Guardar el modelo y los resultados

En `MyDrive/acralis/models/` (los `.pt` no van a git). El `.pt` incluye la configuración
de preprocesamiento para que `/analyze` la reproduzca.

In [ ]:
from datetime import datetime

run_id = f"effb0_derm12345_v2_{datetime.now():%Y%m%d_%H%M}"
torch.save({
    "state_dict": final_model.state_dict(),
    "arch": "efficientnet_b0", "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
    "preprocessing": {"fn": "app.preprocessing.lesion_crop", "margin": CROP_MARGIN, "cache_size": CACHE_SIZE},
    "threshold": threshold, "classes": ["benigno", "sospechoso_melanoma"],
}, MODELS_DIR / f"{run_id}.pt")

def to_py(d: dict) -> dict:
    return {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v)
            for k, v in d.items()}

report = {
    "run_id": run_id, "threshold": threshold, "target_sensitivity_cv": TARGET_SENSITIVITY,
    "test": to_py(metrics),
    "test_ci95": {k: [float(a), float(b)] for k, (a, b) in ci.items()},
    "test_by_subgroup": {k: to_py(v) for k, v in by_subgroup(test_df, "p", threshold).to_dict("index").items()},
    "oof_by_subgroup": {k: to_py(v) for k, v in by_subgroup(train_df, "oof", threshold).to_dict("index").items()},
    "lesion_found_rate": float(df.lesion_found.mean()),
    "config": {"seed": SEED, "img_size": IMG_SIZE, "cache_size": CACHE_SIZE, "crop_margin": CROP_MARGIN,
               "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY,
               "n_folds": N_FOLDS},
    "fold_histories": fh.to_dict("records"),
    "final_history": final_history,
}
(MODELS_DIR / f"{run_id}.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
train_df[["isic_id", "patient_id", "fold", "subgroup", "target", "oof"]].to_csv(MODELS_DIR / f"{run_id}_oof.csv", index=False)
test_df[["isic_id", "patient_id", "subgroup", "target", "p"]].to_csv(MODELS_DIR / f"{run_id}_test.csv", index=False)
print("Guardado en", MODELS_DIR / run_id)